### Initialization

In [1]:
agent_pids = []

### Add Agent

In [2]:
from pathlib import Path
import subprocess

parent_dir = Path.cwd().parent
proc_agent = subprocess.Popen(["node", "main.js"], cwd=str(parent_dir))
print(f"Running (PID={proc_agent.pid})"); agent_pids.append(proc_agent.pid)

Running (PID=28104)


In [3]:
import time
time.sleep(10)

#### (Optional) Kill proc with pid

In [4]:
# import subprocess
# for pid in agent_pids:
#     print(subprocess.run(["taskkill", "/PID", str(pid), "/F"]))

### Construction and Evaluation

In [5]:
import subprocess, shutil, json
from pathlib import Path

# Path to Node script in current directory
script = (Path.cwd() / "send_prompts.js").resolve()
node = shutil.which("node") or "node"

# Python-defined prompts
prompts = [
    "Lay the foundation for a 15x20 block rectangular building.",
    "Build an arched bridge.",
    "Build a simple house.",
]

proc = subprocess.Popen(
    [node, str(script)],
    cwd=str(script.parent),       # Node's process.cwd() equals the JS folder
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,                    # line-buffered
)

# Send prompts to Node (single JSON line) and close stdin
proc.stdin.write(json.dumps({"prompts": prompts}) + "\n")
proc.stdin.close()

SENTINEL = "::ACTION_MAX::"

try:
    for line in proc.stdout:
        line = line.rstrip("\n")
        print(line)  # always mirror Node logs

        # If Node reports the max-numbered file, parse and print it
        if line.startswith(SENTINEL):
            payload_raw = line[len(SENTINEL):]
            try:
                payload = json.loads(payload_raw)
            except json.JSONDecodeError:
                print("[PY] Failed to parse sentinel JSON.")
                continue

            if payload.get("ok") and "path" in payload:
                file_path = Path(payload["path"])
                print(f"\n[PY] Max action file: index={payload.get('index')} name={payload.get('name')}")
                print(f"[PY] Path: {file_path}")

                try:
                    content = file_path.read_text(encoding="utf-8", errors="replace")
                    # print("[PY] --- File Content Begin ---")
                    # print(content)
                    # print("[PY] --- File Content End ---\n")
                except Exception as e:
                    print(f"[PY] Could not read file: {e}")
            else:
                # e.g., dir not found or no js files
                reason = payload.get("reason", "unknown")
                print(f"[PY] No max file reported (reason={reason}).")
finally:
    proc.stdout.close()
    proc.wait()


🧹 Cleared all files under D:\git\mineCEraft\bots\andy\action-code
✅ Connected to MindServer at http://localhost:8080 (socket id=sBGKqVXlIavowbYMAAAD)

➡️ Sending to andy: "Lay the foundation for a 15x20 block rectangular building."
⏳ Waiting for completion keyword (timeout 20 min)...
📨 [andy] I'll start laying the foundation! !newAction("Build a 15x20 rectangular foundation using available blocks")
📨 [andy] Foundation complete! I've laid out a 15x20 stone foundation. The corners are marked and the entire area is filled in. Ready for the next phase of construction! 
✅ Completion detected for "Lay the foundation for a 15x20 block rectangular building.".
::ACTION_MAX::{"ok":true,"index":0,"name":"0.js","path":"D:\\git\\mineCEraft\\bots\\andy\\action-code\\0.js"}

[PY] Max action file: index=0 name=0.js
[PY] Path: D:\git\mineCEraft\bots\andy\action-code\0.js

➡️ Sending to andy: "Build an arched bridge."
⏳ Waiting for completion keyword (timeout 20 min)...
📨 [andy] I'll build an arched bri

In [6]:
# import subprocess, shutil
# from pathlib import Path

# script = (Path.cwd() / "send_prompts.js").resolve()
# node = shutil.which("node") or "node"

# proc = subprocess.Popen(
#     [node, str(script)],
#     cwd=str(script.parent),
#     stdout=subprocess.PIPE,
#     stderr=subprocess.STDOUT,
#     text=True,
#     encoding="utf-8",
#     errors="replace",
#     bufsize=1,
# )

# try:
#     for line in proc.stdout:
#         print(line, end="")
# finally:
#     proc.stdout.close()
#     proc.wait()


### Remove Agent

In [7]:
print(proc_agent.kill())
print("Process terminated.")

None
Process terminated.
